In [ ]:
!pip install -q -U google-genai

from google import genai
import time
import json
import re

GEMINI_API_KEY = "API"
client = genai.Client(api_key=GEMINI_API_KEY)

MODEL_NAME = "gemini-3.5-flash-lite"

In [ ]:
LABELING_PROMPT = """Kamu bertugas sebagai annotator untuk penelitian klasifikasi persepsi publik terhadap peluang kerja penyandang disabilitas tunarungu di Indonesia. Sumber data adalah komentar YouTube dari video-video mengenai penyandang tunarungu yang bekerja atau beraktivitas di ruang publik.

Berikan label PERSEPSI untuk SETIAP komentar. Pilih SALAH SATU dari 2 kategori: "inklusi" atau "eksklusi".

=== DEFINISI KATEGORI ===

INKLUSI — komentar menunjukkan salah satu dari:
- Dukungan terhadap kemandirian, kesempatan kerja, atau kapabilitas setara penyandang disabilitas
- Apresiasi terhadap pihak (perusahaan/individu) yang memberi kesempatan kerja tanpa diskriminasi
- Ekspresi haru, semangat, atau kekaguman yang tulus terhadap pencapaian mereka
- Kritik/penolakan terhadap tindakan diskriminatif pihak lain (membela hak penyandang disabilitas)

EKSKLUSI — komentar menunjukkan salah satu dari:
- Meragukan kemampuan mereka menjalankan pekerjaan tertentu
- Menganggap keberadaan mereka di suatu pekerjaan berbahaya/berisiko
- Membatasi jenis pekerjaan atau ruang gerak yang dianggap pantas untuk mereka
- Menuduh pihak tertentu melakukan diskriminasi TERHADAP penyandang disabilitas (masalahnya nyata terjadi)
- Menuduh momen/pencapaian mereka cuma "settingan"/gimmick/pencitraan yang meremehkan keaslian usaha mereka
- Mengejek, menghina, menstigma, atau memakai kata kasar terhadap mereka
- Komentar di luar topik disabilitas/pekerjaan (lihat poin 4 di bawah)

=== CARA MEMBEDAKAN KASUS AMBIGU ===

1. Keraguan TIDAK HARUS disampaikan dengan kata kasar. Kalimat sopan yang tetap meragukan kemampuan ("apa bisa menjelaskan ke customer yang banyak?") tetap EKSKLUSI.

2. Kata "diskriminasi" bisa muncul di KEDUA arah — perhatikan siapa yang dituduh:
   - "Laura mendiskriminasi tunarungu" → EKSKLUSI (menuduh ada diskriminasi nyata terhadap mereka)
   - "jangan diskriminasi disabilitas, dukung mereka" → INKLUSI (justru menentang diskriminasi, membela mereka)

3. Pujian yang menyiratkan KETERKEJUTAN karena menganggap mereka seharusnya tidak mampu ("padahal keterbatasan gitu bisa kerja kayak orang normal") itu EKSKLUSI meski terdengar seperti pujian, karena menyiratkan ekspektasi rendah.

4. KOMENTAR DI LUAR TOPIK WAJIB "eksklusi" — kalau komentar TIDAK MEMBAHAS disabilitas/pekerjaan penyandang disabilitas sama sekali (contoh: soal merek HP, timestamp video, promosi produk lain, basa-basi generik seperti "mantap"/"keren"/"lucu" tanpa konteks jelas, komentar soal topik lain yang tidak berhubungan), maka LABEL WAJIB "eksklusi". Ini WAJIB, tanpa pengecualian, karena komentar semacam ini keluar dari topik penelitian dan tidak menunjukkan dukungan aktif terhadap inklusi disabilitas yang jadi syarat kategori "inklusi". Kategori "inklusi" HANYA untuk komentar yang secara jelas dan aktif berkaitan dengan topik disabilitas/pekerjaan.

5. EMOJI sebagai penguat konteks (bukan penentu tunggal):
   - 😢/😭 dalam komentar yang mengharukan = terharu (INKLUSI), bukan sedih negatif
   - 😂/🤣 = tertawa biasa (netral, ikuti konteks kata di sekitarnya) ATAU tertawa mengejek (EKSKLUSI) — tergantung kalimatnya
   - ❤️/🙏/👏/😍 = dukungan (INKLUSI)
   - 😡/🤬/👎 = penolakan (EKSKLUSI)

Nilai berdasarkan MAKNA DAN NADA KESELURUHAN KALIMAT. Bahasa bisa informal/singkatan/bahasa daerah — pahami konteksnya secara wajar, sebagaimana manusia Indonesia memahami percakapan sehari-hari.

=== CONTOH KALIBRASI ===

Teks: "bagus ini ide bagi para tuna rungu.. agar mereka hidup tidak merasa tersisih, mereka seharusnya dilatih utk bisa mandiri dan dapat menghidupi dirinya sendiri, yaitu dengan menjadi sopir."
Label: inklusi

Teks: "hebat banget perusahaan yang mau terima teman tuli bekerja dan memberikan kesempatan yang sama"
Label: inklusi

Teks: "kalo supir bisu masih bisa dipahami tapi supir tuli ? ya bahaya nyetir mobil"
Label: eksklusi

Teks: "Kenapa Tunarungu kerja di Matahari? Kalau customer padat apa bisa menjelaskan satu satu?"
Label: eksklusi

Teks: "Laura melakukan diskriminasi kepada kami, tuna rungu dianggap bodoh karena belum punya gelar"
Label: eksklusi

Teks: "jangan sampai ada diskriminasi kepada disabilitas dinegeri kita ini, lihatlah semangat mereka jangan lihat fisiknya"
Label: inklusi

Teks: "Sebenarnya ini trik Matahari biar makin ramai pembeli aja, settingan doang ini mah"
Label: eksklusi

Teks: "salut deh padahal keterbatasan gitu bisa juga kerja kayak orang normal"
Label: eksklusi

Teks: "dongo"
Label: eksklusi

Teks: "semoga byk pengusaha bisa menerima para disabilitas, walau mereka kekurangan fisik tp kemampuan jg tdk diragukan"
Label: inklusi

Teks: "Bahasa terindah yg ingin ku pelajari 😢❤️"
Label: inklusi

Teks: "Hpnya iqoo 13"
Label: eksklusi

Teks: "0:23"
Label: eksklusi

Teks: "keren banget videonya"
Label: eksklusi

Teks: "siapa yang nonton pas lagi rebahan angkat tangan"
Label: eksklusi

FORMAT OUTPUT WAJIB — kembalikan HANYA JSON array valid, tanpa markdown, tanpa penjelasan tambahan apapun. Semua id di data input HARUS muncul di output, tidak boleh ada yang terlewat:
[{{"id": "id1", "label": "inklusi"}}, {{"id": "id2", "label": "eksklusi"}}]

Berikut data yang perlu dilabeli:
{data}
"""

def build_batch_text(batch_df):
    lines = []
    for _, row in batch_df.iterrows():
        text_clean = str(row["text"]).replace("\n", " ").replace('"', "'")
        lines.append(f'id: {row["comment_id"]} | text: "{text_clean}"')
    return "\n".join(lines)

def call_gemini_batch(batch_df, max_retries=3):
    prompt = LABELING_PROMPT.format(data=build_batch_text(batch_df))
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt
            )
            raw_text = response.text.strip()
            raw_text = re.sub(r"^```json\s*|\s*```$", "", raw_text.strip())
            results = json.loads(raw_text)
            return {str(item["id"]): item["label"] for item in results}
        except Exception as e:
            error_str = str(e)
            if "429" in error_str or "RESOURCE_EXHAUSTED" in error_str:
                raise SystemExit("Kuota API habis untuk hari ini.")
            print(f"  Percobaan {attempt+1} gagal: {e}")
            time.sleep(10)
    print("  Batch ini gagal setelah semua percobaan, dilewati.")
    return {}

In [ ]:
df_test = df.head(10)
hasil_test = call_gemini_batch(df_test)
print(hasil_test)

{'Ugy__Xbktb9UbUgv5M94AaABAg': 'eksklusi', 'Ugxmuu_b9zZ0DU_Ptad4AaABAg': 'eksklusi', 'UgwGj1x5QwlQQHqiYO54AaABAg': 'eksklusi', 'UgycSjbhAcoghDks6Uh4AaABAg': 'inklusi', 'Ugy9MRkhly_nWufIUAx4AaABAg': 'inklusi', 'UgxP_WXLwIsreiqPCCZ4AaABAg': 'eksklusi', 'UgxabJrYtBJO6x_aNGl4AaABAg': 'eksklusi', 'Ugz9YaNpJMP93z01WUZ4AaABAg': 'eksklusi', 'UgxQmnr1fbfD3N6me2t4AaABAg': 'eksklusi', 'UgyANk6c3WtXQQJBPq14AaABAg': 'eksklusi'}


In [ ]:
BATCH_SIZE = 40
DELAY_BETWEEN_CALLS = 6
MAX_CALLS_PER_RUN = 480
CHECKPOINT_PATH = "/content/drive/MyDrive/labeling_checkpoint_annotator1_v2.csv"

try:
    df_checkpoint = pd.read_csv(CHECKPOINT_PATH, dtype={"comment_id": str})
    labeled_ids = set(df_checkpoint["comment_id"].astype(str))
    print(f"Melanjutkan dari checkpoint: {len(labeled_ids)} sudah dilabeli")
except FileNotFoundError:
    df_checkpoint = pd.DataFrame(columns=["comment_id", "label_annotator_1"])
    labeled_ids = set()
    print("Belum ada checkpoint, mulai dari awal")

df_remaining = df[~df["comment_id"].isin(labeled_ids)].reset_index(drop=True)
print(f"Sisa yang perlu dilabeli: {len(df_remaining)} dari {len(df)}")

n_batches = (len(df_remaining) // BATCH_SIZE) + 1
calls_made = 0
quota_habis = False

for i in range(n_batches):
    if calls_made >= MAX_CALLS_PER_RUN:
        print(f"\nBerhenti di {calls_made} panggilan (batas aman harian). Lanjutkan besok.")
        break

    start = i * BATCH_SIZE
    end = start + BATCH_SIZE
    batch = df_remaining.iloc[start:end]
    if len(batch) == 0:
        continue

    print(f"Batch {i+1}/{n_batches} ({len(batch)} komentar)...")

    try:
        results = call_gemini_batch(batch)
        calls_made += 1
    except SystemExit as e:
        print(f"\n{e}")
        quota_habis = True
        break

    if results:
        new_rows = pd.DataFrame([
            {"comment_id": cid, "label_annotator_1": label}
            for cid, label in results.items()
        ])
        df_checkpoint = pd.concat([df_checkpoint, new_rows], ignore_index=True)
        df_checkpoint.to_csv(CHECKPOINT_PATH, index=False, encoding="utf-8-sig")

    time.sleep(DELAY_BETWEEN_CALLS)

print(f"\nTotal terlabeli: {len(df_checkpoint)} dari {len(df)}")
if quota_habis:
    print("Kuota API habis — lanjutkan besok.")
elif len(df_checkpoint) < len(df):
    print("Masih ada sisa. Jalankan ulang cell ini.")
else:
    print("Semua data sudah berhasil dilabeli ulang untuk Annotator 1!")

Belum ada checkpoint, mulai dari awal
Sisa yang perlu dilabeli: 13613 dari 13613
Batch 1/341 (40 komentar)...
Batch 2/341 (40 komentar)...
Batch 3/341 (40 komentar)...
Batch 4/341 (40 komentar)...
Batch 5/341 (40 komentar)...
Batch 6/341 (40 komentar)...
Batch 7/341 (40 komentar)...
Batch 8/341 (40 komentar)...
Batch 9/341 (40 komentar)...
Batch 10/341 (40 komentar)...
Batch 11/341 (40 komentar)...
Batch 12/341 (40 komentar)...
Batch 13/341 (40 komentar)...
Batch 14/341 (40 komentar)...
Batch 15/341 (40 komentar)...
Batch 16/341 (40 komentar)...
Batch 17/341 (40 komentar)...
Batch 18/341 (40 komentar)...
Batch 19/341 (40 komentar)...
Batch 20/341 (40 komentar)...
Batch 21/341 (40 komentar)...
Batch 22/341 (40 komentar)...
Batch 23/341 (40 komentar)...
Batch 24/341 (40 komentar)...
Batch 25/341 (40 komentar)...
Batch 26/341 (40 komentar)...
Batch 27/341 (40 komentar)...
Batch 28/341 (40 komentar)...
Batch 29/341 (40 komentar)...
Batch 30/341 (40 komentar)...
Batch 31/341 (40 komentar)..